# 🚀 Fine-tuning Qwen/Qwen2.5-Coder-7B-Instruct cho NL to FOL Translation (V2)

Notebook này thực hiện huấn luyện Fine-Tuning (SFT) mô hình **Qwen/Qwen2.5-Coder-7B-Instruct** để dịch Natural Language sang First-Order Logic (FOL).

## 1. Cài đặt Thư viện

In [1]:
!pip install -q transformers peft trl accelerate datasets evaluate huggingface_hub z3-solver


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


## 2. Khai báo Thư viện & Kiểm tra GPU

In [2]:
import os
import json
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer

# Khai báo import linh hoạt cho DataCollatorForCompletionOnlyLM
try:
    from trl import DataCollatorForCompletionOnlyLM
except ImportError:
    try:
        from trl.trainer import DataCollatorForCompletionOnlyLM
    except ImportError:
        try:
            from trl.trainer.utils import DataCollatorForCompletionOnlyLM
        except ImportError:
            from transformers import DataCollatorForLanguageModeling
            
            class DataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
                def __init__(self, response_template, tokenizer, *args, **kwargs):
                    kwargs.setdefault('mlm', False)
                    super().__init__(tokenizer=tokenizer, *args, **kwargs)
                    self.response_template = response_template
                    self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)
                    
                def torch_call(self, examples):
                    batch = super().torch_call(examples)
                    response_token_ids = self.response_token_ids
                    n = len(response_token_ids)
                    
                    labels = batch['labels'].clone()
                    for i in range(len(examples)):
                        input_ids = batch['input_ids'][i].tolist()
                        idx = -1
                        for j in range(len(input_ids) - n + 1):
                            if input_ids[j:j+n] == response_token_ids:
                                idx = j + n
                                break
                        if idx != -1:
                            labels[i, :idx] = -100
                    batch['labels'] = labels
                    return batch

# Kiểm tra GPU
print("CUDA/ROCm Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))


CUDA/ROCm Available: True
GPU Device Name: 


## 3. Cấu hình Đường dẫn & Nạp Tập dữ liệu

In [3]:
BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
TRAIN_FILE = "./qwen_sft_full_train.jsonl"
VAL_FILE = "./qwen_sft_full_val.jsonl"
OUTPUT_DIR = "./qwen_logic_v2_lora"
max_seq_length = 2048

print("📦 Đang tải tập dữ liệu Logic SFT...")
dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})
print(f"Đã tải thành công!")


📦 Đang tải tập dữ liệu Logic SFT...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Đã tải thành công!


## 4. Khởi tạo Tokenizer & Format Dataset

In [5]:
print("⏳ Đang khởi tạo Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_and_tokenize(example):
    output_texts = []
    for msg_list in example['messages']:
        text = tokenizer.apply_chat_template(msg_list, tokenize=False)
        output_texts.append(text)
    return tokenizer(output_texts, truncation=True, max_length=max_seq_length)

print("📝 Đang tokenize dataset...")
dataset = dataset.map(format_and_tokenize, batched=True, remove_columns=dataset["train"].column_names)


⏳ Đang khởi tạo Tokenizer...
📝 Đang tokenize dataset...


Map:   0%|          | 0/6289 [00:00<?, ? examples/s]

Map:   0%|          | 0/331 [00:00<?, ? examples/s]

## 5. Phân chia Train / Val Split

In [6]:
train_dataset = dataset["train"]
val_dataset = dataset["validation"]

print(f"📊 Train samples: {len(train_dataset)}")
print(f"📊 Val samples: {len(val_dataset)}")


📊 Train samples: 6289
📊 Val samples: 331


## 6. Tải Mô hình Qwen/Qwen2.5-Coder-7B-Instruct

In [7]:
print(f"🔥 Đang tải mô hình {BASE_MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
model.enable_input_require_grads()
model.config.use_cache = False
print("Tải mô hình thành công!")

🔥 Đang tải mô hình Qwen/Qwen2.5-Coder-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Tải mô hình thành công!


## 7. Thiết lập LoRA (Tối ưu cho Reasoning/Z3)

In [8]:
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, peft_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


## 8. Cấu hình Trainer & SFTTrainer

In [9]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    per_device_train_batch_size=4,         
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    bf16=True,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    optim="adamw_torch",
    warmup_ratio=0.05,
    weight_decay=0.05,
    report_to="none"
)

response_template = "<|im_start|>assistant\n"
data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template, 
    tokenizer=tokenizer, 
    mlm=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    data_collator=data_collator
)
print("✅ Sẵn sàng huấn luyện!")

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)
The model is already on multiple devices. Skipping the move to device specified in `args`.


✅ Sẵn sàng huấn luyện!


## 9. Bắt đầu Train

In [10]:
print("🚀 BẤM NÚT FINE-TUNING...")
trainer.train()
print("🎉 Hoàn tất huấn luyện!")

# Lưu mô hình LoRA
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Đã lưu adapter tại: {OUTPUT_DIR}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 BẤM NÚT FINE-TUNING...


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.127800,0.191762,0.197889,0.946883,961888.000000
2,0.065400,0.169965,0.103649,0.958611,1923776.000000
3,0.023800,0.195565,0.062973,0.958804,2885664.000000


🎉 Hoàn tất huấn luyện!
Đã lưu adapter tại: ./qwen_logic_v2_lora


In [11]:
final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"🎉 Adapter đã được lưu an toàn tại: {final_adapter_path}")

🎉 Adapter đã được lưu an toàn tại: ./qwen_logic_v2_lora/final_adapter


In [12]:
from huggingface_hub import HfApi, login

TOKEN = os.environ.get("HF_TOKEN", "")
LORA_REPO_ID = "kotorii1/qwen_logic_v2_lora"

if not TOKEN:
    print("⚠️ Vui lòng điền mã Token Hugging Face hoặc thiết lập biến môi trường HF_TOKEN để upload model!")
else:
    print("🔑 Đang tiến hành đăng nhập vào Hugging Face...")
    login(token=TOKEN)

    print(f"⚡ Đang khởi tạo kho chứa Adapter '{LORA_REPO_ID}'...")
    api = HfApi(token=TOKEN)
    try:
        api.create_repo(
            repo_id=LORA_REPO_ID,
            repo_type="model",
            private=True,
            exist_ok=True
        )
        print("✅ Repository đã được thiết lập thành công.")
    except Exception as e:
        print(f"⚠️ Cảnh báo khi tạo repo (có thể đã tồn tại): {e}")

    print(f"🚀 Đang đẩy toàn bộ LoRA adapter từ {final_adapter_path} lên Hugging Face...")
    api.upload_folder(
        folder_path=final_adapter_path,
        repo_id=LORA_REPO_ID,
        repo_type="model"
    )
    print(f"🎉 Thành công mỹ mãn! Adapter của bạn đã được lưu trữ tại: https://huggingface.co/{LORA_REPO_ID}")

🔑 Đang tiến hành đăng nhập vào Hugging Face...
⚡ Đang khởi tạo kho chứa Adapter 'kotorii1/qwen_logic_v2_lora'...
✅ Repository đã được thiết lập thành công.
🚀 Đang đẩy toàn bộ LoRA adapter từ ./qwen_logic_v2_lora/final_adapter lên Hugging Face...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

🎉 Thành công mỹ mãn! Adapter của bạn đã được lưu trữ tại: https://huggingface.co/kotorii1/qwen_logic_v2_lora
